In [9]:
%pip install scikit-learn


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   -------------------- ------------------- 4.2/8.2 MB 32.1 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 32.2 MB/s  0:00:00
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------- ----------------------- 14.7/36.6 MB 69.9 MB/s eta 0:00:01
   ----------------------------- ---------- 26.7/36.6 MB 65.2 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.6 MB 63.9 MB/s eta 0:00:01
   ---------------------------------------- 36.6/36.6 MB 59.1 MB/s  0:00:00

   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ --------------------------------- 1/6 [scipy]
   ------ ----------------------------

In [1]:
from pathlib import Path
import pandas as pd
import json

BASE = Path.cwd()

print("Working directory:", BASE)
print("Files:", [p.name for p in BASE.iterdir()])

labels = pd.read_csv("labelled_pairs.csv")

display(labels["label"].value_counts().to_frame("Count"))
display(
    (labels["label"].value_counts(normalize=True) * 100)
    .round(2)
    .to_frame("Percentage")
)

display(labels.head())

Working directory: c:\Users\ub02-glab-073\Desktop\data-mining-lab\data_2\data_2
Files: ['commands.ipynb', 'labelled_pairs.csv', 'notices', 'portal_profiles.md', '_truth']


,Count
label,
different,621
same,279


,Percentage
label,
different,69.0
same,31.0


,notice_id_a,notice_id_b,label,adjudicated_by,adjudicated_on
0,N010018,N010020,same,ops6,2025-03-28
1,N007876,N008565,different,ops1,2025-04-30
2,N005451,N005452,same,ops2,2025-03-24
3,N008464,N010231,different,ops6,2025-08-20
4,N000420,N001104,different,ops2,2025-08-23


In [2]:
# TASK 2: Load and inspect the notice corpus

from pathlib import Path
import pandas as pd

notice_files = sorted(Path("notices").glob("*.csv"))

notices = pd.concat(
    [pd.read_csv(f) for f in notice_files],
    ignore_index=True
)

print("Number of notice files:", len(notice_files))
print("Total notices:", len(notices))
print("\nColumns:")
print(notices.columns.tolist())

display(notices.head(5))
display(notices.isnull().sum().to_frame("Missing Values"))

Number of notice files: 8
Total notices: 12000

Columns:
['notice_id', 'portal_id', 'published_at', 'title', 'body', 'estimated_value', 'closing_date']


,notice_id,portal_id,published_at,title,body,estimated_value,closing_date
0,N000001,P136,2024-03-03,Procurement and commissioning of CCTV surveill...,Name of work: Procurement and commissioning of...,76000000,2024-03-20
1,N000009,P011,2024-06-18,Desilting and lining of the bus terminal at Os...,Name of work: Desilting and lining of the bus ...,4530000,2024-07-13
2,N000017,P238,2024-07-28,Upgradation of CCTV surveillance infrastructur...,Name of work: Upgradation of CCTV surveillance...,396500000,2024-08-13
3,N000025,P111,2024-07-16,REHABILITATION OF CCTV SURVEILLANCE INFRASTRUC...,NAME OF WORK: REHABILITATION OF CCTV SURVEILLA...,17900000,2024-08-01
4,N000033,P088,2025-06-18,Tender Notice: Augmentation of the 34 MLD sewa...,Name of work: Augmentation of the 34 MLD sewag...,79080000,2025-07-26


,Missing Values
notice_id,0
portal_id,0
published_at,0
title,0
body,0
estimated_value,0
closing_date,0


In [3]:
# TASK 3: Text and portal profile analysis

notices["text_length"] = (
    notices["title"].fillna("").str.len()
    + notices["body"].fillna("").str.len()
)

print("Text length statistics:")
display(notices["text_length"].describe().to_frame())

print("\nNotices per portal:")
display(
    notices["portal_id"]
    .value_counts()
    .head(20)
    .to_frame("Notice Count")
)

print("\nSample portal formatting:")
print(Path("portal_profiles.md").read_text(encoding="utf-8"))

Text length statistics:


,text_length
count,12000.000000
mean,4607.874833
std,1239.364134
min,1542.000000
25%,3798.000000
50%,4562.500000
75%,5355.000000
max,8223.000000



Notices per portal:


,Notice Count
portal_id,
P094,1426
P002,800
P006,792
P001,778
P003,772
P005,768
P004,755
P020,384
P240,377



Sample portal formatting:
# Portal notes (scraping team, informal)

These are working notes, not a specification. They were written by three
different people over two years. Where they contradict the data, trust
the data -- but they will usually tell you *why* the data looks like it does.

## The nodal aggregators -- read this one first

`P001` `P002` `P003` `P004` `P005` `P006` are not procuring entities. They
are aggregation services that re-publish notices on behalf of departments,
and between them they account for a large fraction of everything we scrape.

Two things about them that have bitten us:

1. **They paste the same legal preamble onto every single notice.** P001,
   P002 and P005 use the ~1,400 character 'NATIONAL PROCUREMENT AGGREGATION
   SERVICE' block. P003, P004 and P006 use the 'STATE PROCUREMENT CELL'
   block, which is about the same size. We strip nothing -- the body column
   is exactly what the page contained.
2. **A short notice from one of these portals is mo

In [4]:
# TASK 4: Inspect actual labeled notice pairs

import pandas as pd
from pathlib import Path

labels = pd.read_csv("labelled_pairs.csv")

notice_files = sorted(Path("notices").glob("*.csv"))
notices = pd.concat(
    [pd.read_csv(f) for f in notice_files],
    ignore_index=True
)

# Select a few labeled same and different pairs
sample_pairs = pd.concat([
    labels[labels["label"] == "same"].head(3),
    labels[labels["label"] == "different"].head(3)
])

notice_lookup = notices.set_index("notice_id")

for _, pair in sample_pairs.iterrows():
    a = notice_lookup.loc[pair["notice_id_a"]]
    b = notice_lookup.loc[pair["notice_id_b"]]

    print("=" * 80)
    print("LABEL:", pair["label"])
    print("\nNOTICE A:", pair["notice_id_a"])
    print("Portal:", a["portal_id"])
    print("Title:", a["title"])
    print("Body:", str(a["body"])[:700])

    print("\nNOTICE B:", pair["notice_id_b"])
    print("Portal:", b["portal_id"])
    print("Title:", b["title"])
    print("Body:", str(b["body"])[:700])

LABEL: same

NOTICE A: N010018
Portal: P004
Title: Supply and installation of the check dam on the Sone near Banaskantha
Body: STATE PROCUREMENT CELL -- CONSOLIDATED TENDER BULLETIN
GENERAL INSTRUCTIONS TO BIDDERS (REPRODUCED IN FULL IN EVERY BULLETIN ENTRY)

(a) Tender documents can be downloaded from the portal after payment of the
    prescribed tender processing fee through the integrated payment gateway.
(b) Earnest money deposit shall be furnished in the form of a demand draft,
    fixed deposit receipt, or bank guarantee from a scheduled commercial bank,
    drawn in favour of the officer inviting the tender and payable at par.
(c) The bidder shall enclose self attested copies of the permanent account
    number, goods and services tax registration certificate, contractor
    registration certificate, and aud

NOTICE B: N010020
Portal: P008
Title: Tender Notice: Corrigendum - Supply and installation of the check dam on the Sone near Banaskantha
Body: Name of work: Supply and ins

In [11]:
# TASK 4: Compare two text similarity methods

import pandas as pd
import re
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

# Load labeled pairs
labels = pd.read_csv("labelled_pairs.csv")

# Load notices
files = sorted(Path("notices").glob("*.csv"))
notices = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

notice_lookup = notices.set_index("notice_id")

# Prepare labeled examples
pairs = labels.copy()

def get_text(notice_id):
    row = notice_lookup.loc[notice_id]
    return str(row["title"]) + " " + str(row["body"])

pairs["text_a"] = pairs["notice_id_a"].apply(get_text)
pairs["text_b"] = pairs["notice_id_b"].apply(get_text)

# Method 1: word-level TF-IDF
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=20000,
    stop_words="english"
)

all_text = pd.concat([pairs["text_a"], pairs["text_b"]])
word_vectorizer.fit(all_text)

a_word = word_vectorizer.transform(pairs["text_a"])
b_word = word_vectorizer.transform(pairs["text_b"])

pairs["word_similarity"] = [
    cosine_similarity(a, b)[0, 0]
    for a, b in zip(a_word, b_word)
]

# Method 2: character-level TF-IDF
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=20000
)

char_vectorizer.fit(all_text)

a_char = char_vectorizer.transform(pairs["text_a"])
b_char = char_vectorizer.transform(pairs["text_b"])

pairs["char_similarity"] = [
    cosine_similarity(a, b)[0, 0]
    for a, b in zip(a_char, b_char)
]

# Compare score distributions
summary = pairs.groupby("label")[
    ["word_similarity", "char_similarity"]
].agg(["mean", "median", "min", "max"])

display(summary)

# Show representative examples
display(
    pairs[
        [
            "notice_id_a",
            "notice_id_b",
            "label",
            "word_similarity",
            "char_similarity"
        ]
    ].head(20)
)

word_similarity                              char_similarity  \
                     mean    median       min      max            mean   
label                                                                    
different        0.169143  0.153552  0.048625  0.61607        0.457746   
same             0.880747  0.905338  0.598240  1.00000        0.701144   

                                         
             median       min       max  
label                                    
different  0.419056  0.116237  0.974880  
same       0.757978  0.179844  0.999853

,notice_id_a,notice_id_b,label,word_similarity,char_similarity
0,N010018,N010020,same,0.650831,0.618032
1,N007876,N008565,different,0.171307,0.251299
2,N005451,N005452,same,0.765583,0.791248
3,N008464,N010231,different,0.114181,0.191455
4,N000420,N001104,different,0.158647,0.486567
5,N004781,N005219,different,0.057032,0.288008
6,N007021,N010547,different,0.248362,0.738730
7,N004413,N008734,different,0.129058,0.148222
8,N007783,N007784,same,0.712019,0.871487
9,N001104,N001107,same,0.681732,0.511493


In [12]:

# TASK 5: MinHash estimation accuracy on labeled pairs

import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

K = 128
RNG = np.random.default_rng(42)

def tokens(text):
    text = str(text).lower()
    return set(re.findall(r"[a-z0-9]+", text)) - set(ENGLISH_STOP_WORDS)

def jaccard(a, b):
    union = a | b
    return len(a & b) / len(union) if union else 1.0

# Stable hash functions for reproducible MinHash
PRIME = 4294967311
A = RNG.integers(1, PRIME, size=K, dtype=np.int64)
B = RNG.integers(0, PRIME, size=K, dtype=np.int64)

def signature(token_set):
    if not token_set:
        return np.full(K, PRIME, dtype=np.int64)

    hashes = np.array([
        abs(hash(token)) % PRIME for token in token_set
    ], dtype=np.int64)

    return np.array([
        np.min((a * hashes + b) % PRIME)
        for a, b in zip(A, B)
    ])

# Use the labeled pairs and notice_lookup created earlier
rows = []

for _, p in labels.iterrows():
    ta = tokens(get_text(p["notice_id_a"]))
    tb = tokens(get_text(p["notice_id_b"]))

    exact = jaccard(ta, tb)
    sig_a = signature(ta)
    sig_b = signature(tb)
    estimate = np.mean(sig_a == sig_b)

    rows.append({
        "label": p["label"],
        "exact_jaccard": exact,
        "estimated_jaccard": estimate,
        "absolute_error": abs(exact - estimate)
    })

estimation = pd.DataFrame(rows)

display(estimation.groupby("label")[
    ["absolute_error"]
].agg(["mean", "median", "max"]))

print("Overall MAE:", estimation["absolute_error"].mean())
print("Overall maximum error:", estimation["absolute_error"].max())

absolute_error                    
                    mean    median       max
label                                       
different       0.033531  0.030442  0.121094
same            0.032777  0.025281  0.125924

Overall MAE: 0.0332971121612258
Overall maximum error: 0.12592405913978494
